# CSA XE continuation — epochs11–15, two GPUs
**T4×2, Internet ON, Save Version → Save & Run All.** Add Input:
- `csa-full-roi-karpathy-v134`
- `csa-semantic-banks-v130`
- `csa-full-xe10-v139` (latest `checkpoint.bin` and `best.bin`)

Paths prefilled. No API token or mounted review ZIP. Missing/extracted receipts for original139 are restored from pinned metadata only after exact SHA verification of both binaries; inputDataset untouched. Mandatory resume from latest epoch10/177110updates; never from best9/fresh weights. Interrupted15run still needs its own verified receipts.
Five additional fullTRAIN epochs:88555updates; absolute endpoint15=265665updates. Same model/Adam/RNG/data/global32/FP32; LR remains1e-4, no warmup restart/SCST/architecture change. TwoT4DDP, ordinary features; six current-environment gradient/loss checks before actual training.
Full5000val NLL/CIDEr and originalbeam5/max20 after every epoch. Keep oldbest9 unless a new validationCIDEr exceeds it (earlier ties). No TEST.
Then CPUscore savedbest9 and11–15 CIDEr/BLEU4/ROUGE-L/CHAIRs/i/recall/counts/length, with oldbest9 metric replay. Selection is stillCIDEr, not post-hoc CHAIR. Training and CPUscore completion reported separately; scorer failure does not erase training checkpoint.
Measured old10epoch worker≈6h;5extra may take roughly3h plussetup/scoring, not a guarantee.6h parent training/preparation cap +40minCPUscorecap.23h reported quota; elapsedtime and billing are different.
Preserve ENTIRE `csa_full_xe_15` folder (latest,best,receipts,validation captions). For interruption, save it asDataset and change RESUME_ROOT to that entire folder in the SAME pinned notebook/fresh output path. Never substitute a new source revision for an interrupted15run without audit.
Send `csa_full_xe_15_review.zip`. Local agent checks are metadata/NumPy/static only; CUDA/real resume/metrics are user-run pending.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil
from pathlib import Path
FULL_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-full-roi-karpathy-v134')
BANK_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-semantic-banks-v130')
RESUME_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-full-xe10-v139')
OUTPUT=Path('/kaggle/working/csa_full_xe_15')
WALL_HOURS=6.0 # Stop cap for preparation/training, not ETA; CPU quality scoring has a separate cap.
COMMIT='04fee9f39e6b7c1502b9c01fe0afe10a647490fb'
HASHES={'run_sharing_benchmark.py': '51706ae5cbd96fb0484cb0487a05f14f1b34f4332bd9cb57a299d7d3d5a14d18', 'sharing_data.py': '68a065f5c055a55b41eae0ea554fbb7f8fd1124e8f5422679fcb1d4878f14829', 'full_cache_data.py': '58cf4d08310fe03e01441af2ddeb12fd3404f59e82259194a32fe98c8b002bc1', 'model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'pilot_data.py': '0edab33c7ad7cc54c28efdf6007f23fa6dc4547bdda32e5c0d7d888a3369394d', 'package_full_roi_dataset.py': 'be3e7d0f96add450a5295add6466e19977da90f486e5e726e3ea40a505cceecb', 'SAVE_FULL_ROI_V134_PACKAGE.json': 'a73636ca75806d6b1d7b4d88f80f3ce1c59b3061eb3e9a5e730097ddd89c505c', 'SAVE_BANK_V130_PACKAGE.json': '826ba36628d6587860e2d548e950ea953bd4bbb8c0a8dc64393089dbc1e058e1', 'REAL_BANK_INPUT_PACKAGE.json': 'c60902e04ea15cf90f652e384937bb317e288753f13b8ede6ab5ef89f8d97129', 'lemma_correction_review_text/bank_words_corrected.json': 'cec76a0229f30361235a4b64e9712ac6bb2e79fc62a9241079a7af3fa95d13ac', 'DDP_SHARING_BENCHMARK_PROTOCOL.md': '6537e5e44fb8eb1504c4234b87d4e4e87929cf80fed3884f3caa1ef09468b2f1', 'run_full_xe.py': '037ee85540ef3bd61b459e68dd1bae9328cfc80f901e53dcfea21cbd1471eb84', 'full_xe_data.py': 'a7a5a3b4d70c9d3a357aa464dfe397a6cba8d851e33065299fb98721ddca7135', 'full_xe_decode.py': 'b8e618559feec6361cfb3b4619899e4968ab71ab67e113ff38b6d3cd99cdc235', 'test_full_xe.py': 'aee46db054fd18bcf9306d8d8240414d67ca39dccb74d7ab0337e5d9ef7cb1fa', 'FULL_XE_10_PROTOCOL.md': 'b52d4081cf0657c7c7fba8a00def7efbff928b4979180e08b4e552cc3983a70c', 'make_xe15_notebook.py': 'af0b8c0483b735444f2bcfc3ba4e9f9aef85f9a8f0dcc131041a22d593c798b2', 'xe15_resume_data.py': 'cd27338248f70b530db3eaab1ae76edf3661a6265313d5fe18243b415e111b3f', 'test_xe15_resume.py': '111b3ffaf6a7c4128f5772049e5c77061275ad4eb80725661c7b2a54f99bb791', 'XE15_CONTINUATION_PROTOCOL.md': 'b2f65a81ab44ea03566a357710722e138b5e7199fe43194a02bd8585c4c4eafa', 'score_xe15_quality.py': '2894dde0eafe0690b778cc1baa9e74fadd6bf79abc61f3d792c00594ab733b4d', 'full_xe10_review_text/input_identity.json': 'b7d1f08ecc045ea6779c5c04776b52d5b92933a88d58fe792b3f897c5cd98c5b', 'SAVE_XE139_PACKAGE.json': '452ee35118a3d0285bc7f522bb8c8fc7a0e3e0dac70cac9c2333e7583cd4e423', 'run_best9_decode.py': 'fdc6b027ac13d510be72aa25a963eb89e5467e09f179bfaf0756469aef77a648', 'saved_quality_data.py': 'f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e', 'best9_saved_text.json.gz': '42594e90ec2f47646e7e68347a17471213038bc182814ca9d29ea374e8010861', 'BEST9_SAVED_TEXT_PACKAGE.json': '898d67c3eead66c89f5d2c091b24e5e3eb44d884e86550b2b26f45d6708032e5', 'saved_quality_review_text/comparison.json': '43519017eaaf9006264011f9994255d030d817dc9b44d49553ef346f302051e9', 'full_xe10_review_text/checkpoint_receipt.json': 'a46cc05fe4af3fe52699dc0a557a816d33cbb08fa69694eefcb1e84cfcc52f68', 'full_xe10_review_text/best_receipt.json': '9bc44a09f17074b5087cbf13e7b13461048e70e0aaee3f89a12f6834517599f9'}
if not all(p.is_dir() for p in (FULL_ROOT,BANK_ROOT,RESUME_ROOT)):raise FileNotFoundError('Add fullROI134, semantic-bank130 and checkpoint139 Datasets.')
if OUTPUT.exists():raise RuntimeError('Preserve existing output; use a fresh session/output folder.')
if min(shutil.disk_usage('/tmp').free,shutil.disk_usage('/kaggle/working').free)<6*1024**3:raise RuntimeError('Need6GiB free disk.')
OUTPUT.mkdir(parents=True);SOURCE=Path(tempfile.mkdtemp(prefix='csa-xe11-15-'));STAGE=SOURCE/'stage';LOG=OUTPUT/'notebook.log'
os.environ['OMP_NUM_THREADS']='1'
for n,h in HASHES.items():
    raw=urllib.request.urlopen(f'https://raw.githubusercontent.com/Supzxjee/Image_Captioning/{COMMIT}/experiments/csa_paper/{n}',timeout=90).read()
    if hashlib.sha256(raw).hexdigest()!=h:raise ValueError('Changed source:'+n)
    p=SOURCE/n;p.parent.mkdir(parents=True,exist_ok=True);p.write_bytes(raw)
(OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=COMMIT,sha256=HASHES),indent=2),encoding='utf-8')


In [ ]:
def run(args, timeout=1800):
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('CSA continuation active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Continuation phase cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
training_complete=False;quality_complete=False;parent_deadline=time.monotonic()+WALL_HOURS*3600
common=['--bank-root',BANK_ROOT,'--stage',STAGE,'--output',OUTPUT,'--target-epochs','15']
try:
    # Fail missing/changed checkpoints before staging/training. Recursive directory lookup accepts nested mount folders.
    sys.path.insert(0,str(SOURCE))
    from xe15_resume_data import resolve_resume_root
    RESUME_ROOT=resolve_resume_root(RESUME_ROOT,SOURCE/'verified-resume139')
    digest=lambda p:hashlib.file_digest(Path(p).open('rb'),'sha256').hexdigest()
    receipt=json.loads((RESUME_ROOT/'checkpoint_receipt.json').read_text(encoding='utf-8'))
    best=json.loads((RESUME_ROOT/'best_receipt.json').read_text(encoding='utf-8'))
    if digest(RESUME_ROOT/'checkpoint.bin')!=receipt['sha256'] or digest(RESUME_ROOT/'best.bin')!=best['sha256']:raise ValueError('Checkpoint/best bytes differ from receipts')
    if not 177110<=receipt['updates']<=265665:raise ValueError('Expected latest epoch10 or interrupted11–15 output')
    run([sys.executable,'-u',SOURCE/'test_full_xe.py'],timeout=120)
    run([sys.executable,'-u',SOURCE/'test_xe15_resume.py'],timeout=120)
    run([sys.executable,'-m','pip','install','-q','pycocoevalcap==1.2','nltk==3.9.2','future'],timeout=300)
    run([sys.executable,'-m','pip','install','-q','--no-deps','git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'],timeout=300)
    run([sys.executable,'-c',"import nltk;from pattern.en import singularize;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab')"],timeout=300)
    run(['java','-version'],timeout=30)
    CHAIR=SOURCE/'chair'
    run(['git','init',CHAIR],timeout=60)
    run(['git','-C',CHAIR,'remote','add','origin','https://github.com/Maxlinn/CHAIR-metric-standalone.git'],timeout=60)
    run(['git','-C',CHAIR,'fetch','--depth','1','origin','4087a26211aa2339b9a76307cb8f0321ef691d0a'],timeout=300)
    run(['git','-C',CHAIR,'checkout','--detach','FETCH_HEAD'],timeout=60)
    if hashlib.sha256((CHAIR/'chair.pkl').read_bytes()).hexdigest()!='73ce11225febb97df38c5bfad60b3ff6cc25852e9df9a1330673155b4be851c3':raise ValueError('Changed CHAIR cache')


    run([sys.executable,'-u',SOURCE/'run_full_xe.py','prepare','--full-root',FULL_ROOT,*common],timeout=900)
    run([sys.executable,'-u',SOURCE/'run_full_xe.py','metric_preflight',*common],timeout=300)
    sys.path.insert(0,str(SOURCE))
    from run_best9_decode import saved_text
    raw,_=saved_text()
    for n in ('val_epoch09_rank0.json','val_epoch09_rank1.json'):(OUTPUT/n).write_bytes(raw[n])
    # Carry previous extension validation text when resuming an interrupted11–15 run.
    for p in RESUME_ROOT.glob('val_epoch*_rank*.json'):
        if p.name not in ('val_epoch09_rank0.json','val_epoch09_rank1.json'):shutil.copy2(p,OUTPUT/p.name)
    preflight=OUTPUT/'preflight';preflight.mkdir()
    launch=[sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0']
    run([*launch,SOURCE/'run_sharing_benchmark.py','worker','--bank-root',BANK_ROOT,'--stage',STAGE,'--output',preflight],timeout=600)
    if json.loads((preflight/'result.json').read_text(encoding='utf-8'))['preflight_cases']!=6:raise ValueError('Missing six DDP guards')
    remaining=parent_deadline-time.monotonic()
    if remaining<600:raise TimeoutError('Insufficient worker time after preparation')
    run([*launch,SOURCE/'run_full_xe.py','worker',*common,'--resume-root',RESUME_ROOT,'--wall-seconds',remaining-60],timeout=remaining)
    result=OUTPUT/'result.json'
    training_complete=result.is_file() and json.loads(result.read_text(encoding='utf-8'))['status']=='FULL_XE_15_COMPLETE'
    if not training_complete:raise RuntimeError('Training interrupted; preserve ENTIRE folder, resume same pinned notebook with updated RESUME_ROOT')
    run([sys.executable,'-u',SOURCE/'score_xe15_quality.py','--output',OUTPUT,'--chair',CHAIR],timeout=2400)
    quality_complete=json.loads((OUTPUT/'quality_result.json').read_text(encoding='utf-8'))['status']=='XE15_SAVED_QUALITY_COMPLETE'
except BaseException:
    (OUTPUT/'failure.txt').write_text(traceback.format_exc(),encoding='utf-8');raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=training_complete and quality_complete,training_complete=training_complete,
        quality_complete=quality_complete,target_total_epochs=15,requested_added_epochs=5,test_used=False)),encoding='utf-8')
    archive=Path(str(OUTPUT)+'_review.zip')
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUTPUT.rglob('*')):
            if p.is_file() and p.suffix in ('.json','.log','.txt'):z.write(p,str(p.relative_to(OUTPUT)))
    print('SEND REVIEW:',archive,flush=True)
    print('KEEP ENTIRE FOLDER:',OUTPUT,flush=True)
    for n in ('checkpoint.bin','best.bin'):
        if (OUTPUT/n).is_file():print('KEEP:',OUTPUT/n,flush=True)
    print('COMPLETE15 + QUALITY' if training_complete and quality_complete else 'INCOMPLETE: preserve output/logs; do not restart training automatically.',flush=True)
